<a href="https://colab.research.google.com/github/Kaoutarash8/Bert_Arabert/blob/main/Bert_Arabert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Similarity Using BERT

In [1]:
!pip install transformers torch

from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

# BERT anglais (uncased = ignore case)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

def encode_sentence(sentence):
    """Encode une phrase anglaise en vecteur"""
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)

    # Pooling mean (moyenne des tokens)
    attention_mask = inputs["attention_mask"]
    token_embeddings = outputs.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sentence_embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sentence_embedding

def cosine_similarity(vec1, vec2):
    return F.cosine_similarity(vec1, vec2).item()

# Exemples en anglais
text1 = "The cat sleeps on the couch"
text2 = "A cat is napping on the sofa"
text3 = "The car drives fast on the highway"

# Encodage
emb1 = encode_sentence(text1)
emb2 = encode_sentence(text2)
emb3 = encode_sentence(text3)

# Similarités
sim12 = cosine_similarity(emb1, emb2)
sim13 = cosine_similarity(emb1, emb3)

print(f"Similarity '{text1}' vs '{text2}': {sim12:.4f}")
print(f"Similarity '{text1}' vs '{text3}': {sim13:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity 'The cat sleeps on the couch' vs 'A cat is napping on the sofa': 0.9076
Similarity 'The cat sleeps on the couch' vs 'The car drives fast on the highway': 0.5592


# Arabic Text Similarity: mBERT vs AraBERT

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F


mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
mbert_model = AutoModel.from_pretrained("bert-base-multilingual-cased")


arabert_tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")
arabert_model = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv2")

def embed_sentence(text, tokenizer, model):

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)


    attention_mask = inputs["attention_mask"]
    token_embeddings = outputs.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sentence_embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sentence_embedding

def cosine_similarity(emb1, emb2):
    return F.cosine_similarity(emb1, emb2).item()


sentences = [
    "القط نائم على الأريكة",
    "هر صغير يغفو على الكنبة",
    "السيارة تسير بسرعة على الطريق"
]

print("\n📊 مقارنة التشابه بين الجمل")
print("=" * 60)

for i in range(len(sentences)-1):
    for j in range(i+1, len(sentences)):
        # mBERT
        emb1_mbert = embed_sentence(sentences[i], mbert_tokenizer, mbert_model)
        emb2_mbert = embed_sentence(sentences[j], mbert_tokenizer, mbert_model)
        sim_mbert = cosine_similarity(emb1_mbert, emb2_mbert)

        # AraBERT
        emb1_arabert = embed_sentence(sentences[i], arabert_tokenizer, arabert_model)
        emb2_arabert = embed_sentence(sentences[j], arabert_tokenizer, arabert_model)
        sim_arabert = cosine_similarity(emb1_arabert, emb2_arabert)

        print(f"\nالجملة 1: {sentences[i]}")
        print(f"الجملة 2: {sentences[j]}")
        print(f"  🟦 mBERT    : {sim_mbert:.4f}")
        print(f"  🟩 AraBERT  :  {sim_arabert:.4f}")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 مقارنة التشابه بين الجمل

الجملة 1: القط نائم على الأريكة
الجملة 2: هر صغير يغفو على الكنبة
  🟦 mBERT    : 0.7706
  🟩 AraBERT  :  0.6708

الجملة 1: القط نائم على الأريكة
الجملة 2: السيارة تسير بسرعة على الطريق
  🟦 mBERT    : 0.6127
  🟩 AraBERT  :  0.6237

الجملة 1: هر صغير يغفو على الكنبة
الجملة 2: السيارة تسير بسرعة على الطريق
  🟦 mBERT    : 0.5610
  🟩 AraBERT  :  0.5512
